# Bug hunt: five bugs in a working-looking language-model project

Every cell below runs without an error and prints a plausible number. Each of the five stages contains **one planted mistake**, of the kind the course taught you to spot (all of them run on a CPU):
a result that is too good, or too bad, or that changes when it should not. Your job is to find it, say what is wrong, and fix it.

For each stage: read the code, look at the result, and decide what the number should have been. Then write your diagnosis in the markdown cell under it.
The answer key is in `solutions/bug_hunt_answers.ipynb`. It names each bug, fixes it and shows the corrected numbers.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

## Stage 1: a tiny language model

A one-layer attention model learns to predict the next token. The data are random tokens, so nothing can be predicted: the best possible loss is ln 20 = 3.0 nats.

In [ ]:
V, T, D = 20, 16, 32
class TinyLM(nn.Module):
    def __init__(self, causal):
        super().__init__(); self.causal = causal
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(T, D)
        self.q, self.k, self.v, self.out = nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, D), nn.Linear(D, V)
    def forward(self, ids):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1]))
        scores = self.q(x) @ self.k(x).transpose(1, 2) / D ** 0.5
        weights = scores.softmax(-1)                                              # every position reads every position
        return self.out(weights @ self.v(x))
def fit(causal, steps=500):
    torch.manual_seed(0); m = TinyLM(causal); opt = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(steps):
        ids = torch.randint(0, V, (64, T))
        loss = F.cross_entropy(m(ids)[:, :-1].reshape(-1, V), ids[:, 1:].reshape(-1)); opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()
print("final training loss:", round(fit(False), 2))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 2: scoring a sentence with GPT-2

We measure GPT-2's perplexity on an ordinary English sentence. A good small model should be somewhere around 50 to 300.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "The robot put the battery in the charger because it was empty, and then it waited quietly for the light to turn green."
def perplexity(tokenizer):
    ids = tokenizer(text, return_tensors="pt")["input_ids"]
    with torch.no_grad(): loss = model(ids, labels=ids).loss.item()
    return float(np.exp(loss))
print("perplexity:", round(perplexity(tok)))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 3: batched generation

We generate a continuation for two prompts of different lengths in one batch, and compare each with generating it alone. They should be identical (greedy decoding).

In [ ]:
gpt = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tk = AutoTokenizer.from_pretrained("gpt2"); tk.pad_token = tk.eos_token; tk.padding_side = "right"
prompts = ["The battery", "After the robot had finished charging all of its batteries, it"]
def generate(batch_prompts):
    enc = tk(batch_prompts, return_tensors="pt", padding=True)
    with torch.no_grad(): out = gpt.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tk.eos_token_id)
    return [tk.decode(o[enc["input_ids"].shape[1]:]) for o in out]
alone = [generate([p])[0] for p in prompts]; together = generate(prompts)
print("batch equals single for each prompt:", [a == b for a, b in zip(alone, together)])

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 4: evaluating a language model

A model that remembers its training lines is trained on 1,000 log lines and evaluated on 150 held-out lines. We report the perplexity.

In [ ]:
import random, math
from collections import Counter
random.seed(0)
lines = [f"robot R-{random.randint(1, 30)} reported battery {random.randint(1, 99)} percent at charger C{random.randint(1, 5)}" for _ in range(1000)]
pool = lines + random.sample(lines, 150)                                     # the pool was assembled from two exports that overlap
random.shuffle(pool); train_lines, eval_lines = pool[:1000], pool[1000:]
def memory_perplexity(train, test):
    # a model that remembers its training lines: the next word is predicted from the whole prefix of the line (add-0.01 smoothing)
    prefix, nxt = Counter(), Counter()
    for l in train:
        w = l.split()
        for i in range(len(w)): prefix[tuple(w[:i])] += 1; nxt[tuple(w[:i + 1])] += 1
    vocab = len({x for l in train for x in l.split()}) + 1
    total, n = 0.0, 0
    for l in test:
        w = l.split()
        for i in range(len(w)): total += -math.log((nxt[tuple(w[:i + 1])] + 0.01) / (prefix[tuple(w[:i])] + 0.01 * vocab)); n += 1
    return math.exp(total / n)
print("held-out perplexity:", round(memory_perplexity(train_lines, eval_lines), 2), "| lines of the evaluation set that also occur in the training set:", sum(l in set(train_lines) for l in eval_lines))

**Your diagnosis:** *(what is wrong, and how would you fix it?)*

## Stage 5: preparing chat data for fine-tuning

We build one supervised example from a question and its answer, and mark which tokens carry loss. The model should learn to write the answer and then the end-of-turn marker.

In [ ]:
qtok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
question, answer = "What is the capital of France?", "Paris."
prompt_text = f"User: {question}\nAssistant: "
full_text = prompt_text + answer                                                       # hand-written template
ids = qtok(full_text)["input_ids"]; start = len(qtok(prompt_text)["input_ids"])
loss_tokens = qtok.convert_ids_to_tokens(ids[start:])
print("tokens with loss:", loss_tokens, "| the end-of-turn marker is trained:", "<|im_end|>" in loss_tokens)

**Your diagnosis:** *(what is wrong, and how would you fix it?)*